In [ ]:
# 24.1 — Eigenfaces for Face Recognition / 特征脸用于人脸识别

**Chapter 24 — File 1 of 1 / 第24章 — 第1个文件（共1个）**

## Summary / 总结

Implement face recognition using eigenfaces (PCA-based approach). Demonstrates how PCA can extract the most discriminative features from high-dimensional image data and enable efficient face recognition.

使用特征脸（基于PCA的方法）实现人脸识别。演示PCA如何从高维图像数据中提取最有区别的特征并实现高效的人脸识别。

## Eigenfaces Concept / 特征脸概念

- **Idea**: Faces lie on a subspace; PCA extracts basis vectors (eigenfaces)
- **Advantage**: Compact representation; efficient matching
- **Dataset**: AT&T Face Database (from zip file; 40 subjects, 10 images each)

## Step 1 — Import Libraries / 导入库

In [ ]:
# Import libraries for face recognition
# 导入人脸识别库
import zipfile
import numpy as np
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances
import matplotlib.pyplot as plt

## Step 2 — Load Face Data from ZIP / 从ZIP加载人脸数据

In [ ]:
# Code pattern for loading AT&T face database
# 加载AT&T人脸数据库的代码模式

example_code = '''
# Extract and load face images from ZIP archive
def load_faces_from_zip(zip_path):
    """
    Load AT&T face database from ZIP file
    每个文件夹代表一个人，包含10个图像
    """
    faces = []
    labels = []
    subject_id = 0
    
    with zipfile.ZipFile(zip_path, 'r') as zf:
        # List files in archive
        file_list = zf.namelist()
        
        # Group by person (folder)
        for person_dir in sorted(set(f.split('/')[0] for f in file_list)):
            # Load all images for this person
            person_images = [f for f in file_list if f.startswith(person_dir)]
            
            for img_path in sorted(person_images):
                if img_path.endswith(('.pgm', '.jpg', '.png')):
                    # Read image from ZIP
                    with zf.open(img_path) as f:
                        img = Image.open(f)
                        # Flatten to 1D vector
                        face_vector = np.array(img).flatten()
                        faces.append(face_vector)
                        labels.append(subject_id)
            subject_id += 1
    
    return np.array(faces), np.array(labels)

# Load the data
X, y = load_faces_from_zip('att_faces.zip')
print(f"Face dataset shape: {X.shape}")
print(f"  Samples: {X.shape[0]} face images")
print(f"  Features: {X.shape[1]} pixels")
print(f"  Subjects: {len(np.unique(y))}")
'''

print("Loading Face Data from ZIP:")
print("="*70)
print(example_code)

## Step 3 — Extract Eigenfaces using PCA / 使用PCA提取特征脸

In [ ]:
# Extract eigenfaces
# 提取特征脸

example_code = '''
# Apply PCA to extract eigenfaces
# Each eigenface is a direction of variation across all faces
n_components = 50  # Number of eigenfaces to keep
pca = PCA(n_components=n_components)
faces_pca = pca.fit_transform(X)

print(f"Explained variance by eigenfaces:")
cumsum = np.cumsum(pca.explained_variance_ratio_)
print(f"  First eigenface: {pca.explained_variance_ratio_[0]:.4f}")
print(f"  First 10 eigenfaces: {cumsum[9]:.4f}")
print(f"  First 50 eigenfaces: {cumsum[-1]:.4f}")

# Visualize eigenfaces as images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    # Reshape eigenface back to image
    eigenface = pca.components_[i].reshape(46, 56)  # AT&T image size
    ax.imshow(eigenface, cmap='gray')
    ax.set_title(f'Eigenface {i+1}')
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"\nEigenfaces represent fundamental face patterns")
print(f"Any face can be reconstructed as: face ≈ mean_face + Σ(weight_i × eigenface_i)")
'''

print("Extracting Eigenfaces (PCA):")
print("="*70)
print(example_code)

## Step 4 — Face Recognition by Distance / 通过距离的人脸识别

In [ ]:
# Face recognition using eigenface distance
# 使用特征脸距离的人脸识别

example_code = '''
def recognize_face(test_face, training_faces_pca, training_labels, pca_model):
    """
    Recognize a test face by finding closest match in training set
    通过在训练集中找到最接近的匹配来识别测试人脸
    """
    # Project test face to eigenface space
    test_face_pca = pca_model.transform(test_face.reshape(1, -1))
    
    # Compute distances to all training faces
    distances = euclidean_distances(test_face_pca, training_faces_pca)[0]
    
    # Find nearest neighbor
    nearest_idx = np.argmin(distances)
    nearest_label = training_labels[nearest_idx]
    nearest_distance = distances[nearest_idx]
    
    return nearest_label, nearest_distance

# Test on a face
test_idx = 0
test_face = X[test_idx]
true_label = y[test_idx]

# Recognize
predicted_label, distance = recognize_face(test_face, faces_pca, y, pca)

print(f"Test face: Subject {true_label}")
print(f"Predicted: Subject {predicted_label}")
print(f"Eigenface distance: {distance:.4f}")
print(f"Correct prediction: {true_label == predicted_label}")
'''

print("Face Recognition by Distance:")
print("="*70)
print(example_code)

## Step 5 — Evaluate Recognition Performance / 评估识别性能

In [ ]:
# Evaluate recognition accuracy
# 评估识别精度

example_code = '''
# Evaluate on all test images
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Train PCA
pca = PCA(n_components=50)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

# Recognize all test faces
predictions = []
for test_face in X_test:
    pred_label, _ = recognize_face(test_face, X_train_pca, y_train, pca)
    predictions.append(pred_label)

# Evaluate
accuracy = accuracy_score(y_test, predictions)
print(f"Face Recognition Accuracy: {accuracy:.2%}")
print(f"\nConfusion matrix (subset):")
cm = confusion_matrix(y_test, predictions)
print(cm[:5, :5])  # Show top-left corner
'''

print("Evaluating Recognition Performance:")
print("="*70)
print(example_code)

In [ ]:
## Learning Notes / 学习笔记

- **Math Essence**: Eigenfaces exploits the fact that human faces occupy a subspace of image space. PCA finds this subspace, enabling compact representation (50 components << original pixel dimensions). Recognition is efficient: just compute distance in low-dimensional space.
  
  **数学本质**：特征脸利用人脸占据图像空间子空间的事实。PCA找到此子空间，实现紧凑表示。识别在低维空间中计算距离很有效。

- **ML Application**: (1) Eigenfaces was the leading face recognition approach before deep learning (1980s-2000s), (2) Still relevant for understanding dimensionality reduction and its practical applications, (3) Modern systems use CNNs (ConvNets) for better performance, but eigenfaces principles remain important, (4) Advantages: interpretable (can visualize eigenfaces), efficient. Disadvantages: lighting/pose sensitive, lower accuracy than modern methods.
  
  **ML应用**：(1) 特征脸在深度学习前是领先的人脸识别方法，(2) 仍与理解降维相关，(3) 现代系统使用CNNs获得更好性能，(4) 优势：可解释、高效。劣势：对光照/姿态敏感。

➡️ **Next / 下一步**: `../appendix_02/1_versions.ipynb` — Library versions

## Complete Code / 完整代码一览

In [ ]:
# --- Import Section / 导入部分 ---
import zipfile
import numpy as np
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances
import matplotlib.pyplot as plt

# --- Load Face Data from ZIP / 从ZIP加载人脸数据 ---
# with zipfile.ZipFile('att_faces.zip', 'r') as zf:
#     file_list = zf.namelist()
#     for filename in file_list[:5]:
#         print(filename)

# --- PCA for Eigenfaces / 特征脸的PCA ---
# pca = PCA(n_components=50)
# faces_pca = pca.fit_transform(X)
# print(f"Explained variance: {pca.explained_variance_ratio_.sum():.3f}")

# --- Face Recognition / 人脸识别 ---
# test_face_pca = pca.transform(test_face.reshape(1, -1))
# distances = euclidean_distances(test_face_pca, faces_pca)
# nearest = np.argmin(distances)
# print(f"Predicted person: {nearest}")